# Micro-BERT QA Training on Google Colab

This notebook trains the custom 1.8M-parameter Micro-BERT for extractive Question Answering.

**Runtime:** GPU (Tesla T4) — ~5-8 minutes for full SQuAD training

---

## Step 1: Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 2: Mount Google Drive (Optional)

Save your trained model to Google Drive so it persists after the session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create a folder for your project
import os
SAVE_DIR = "/content/drive/MyDrive/Micro-BERT-QA"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Models will be saved to: {SAVE_DIR}")

## Step 3: Clone the Repository

In [ ]:
%cd /content
!git clone https://github.com/YOUR_USERNAME/BERT-Question-Answering-Project.git
%cd BERT-Question-Answering-Project
!ls -la

## Step 4: Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

# Verify GPU is being used by PyTorch
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"Using: {torch.cuda.get_device_name(0)}")

## Step 5: Train Micro-BERT on Full SQuAD

**Options:**
- `dataset=squad` → Full SQuAD 1.1 (~87k examples, **recommended**)
- `dataset=data/sample_squad.json` → Sample data (7 examples, for quick testing)

**Expected time:**
- Sample data (200 epochs): ~30 seconds
- Full SQuAD (3 epochs): ~5-8 minutes on T4 GPU

In [ ]:
# OPTION A: Full SQuAD training (RECOMMENDED)
!python train.py --dataset squad --epochs 3 --batch_size 16 --learning_rate 3e-5 --output_dir checkpoints/micro-bert-qa

# OPTION B: Quick sample training (for testing)
# !python train.py --epochs 200 --batch_size 4 --learning_rate 1e-3 --output_dir checkpoints/micro-bert-qa

## Step 6: Evaluate the Model

In [ ]:
!python run_evaluation.py --dataset data/sample_squad.json --output results/eval_results.json

## Step 7: Save Model to Google Drive

In [ ]:
import shutil
import os

# Copy checkpoint to Drive
if os.path.exists('checkpoints/micro-bert-qa'):
    shutil.copytree('checkpoints/micro-bert-qa', f'{SAVE_DIR}/micro-bert-qa', dirs_exist_ok=True)
    print(f"Model saved to {SAVE_DIR}/micro-bert-qa")
else:
    print("No checkpoint found!")

# Copy results
if os.path.exists('results'):
    shutil.copytree('results', f'{SAVE_DIR}/results', dirs_exist_ok=True)
    print(f"Results saved to {SAVE_DIR}/results")

## Step 8: Test Inference (Interactive)

In [ ]:
from src.model import BERTQA

# Load trained model
model = BERTQA()
info = model.info()
print(f"Model: {info['model_name']}")
print(f"Parameters: {info['num_parameters']:,}")
print(f"Device: GPU")

# Test
context = """The Amazon rainforest, often referred to as the 'lungs of the Earth,' produces about 20% of the world's oxygen. It spans across nine countries in South America, with the majority located in Brazil. The forest is home to approximately 10 million species of animals, plants, and insects."""
question = "What percentage of the world's oxygen does the Amazon produce?"

result = model.answer(question, context)
print(f"\nQuestion: {question}")
print(f"Answer: {result['answer']}")
print(f"Confidence: {result['score']:.4f}")

---

## BONUS: Train BERT-large on Colab

If you want to compare, you can also fine-tune BERT-large (340M) on Colab:

```python
# This takes ~1-1.5 hours on T4 GPU
!python run_evaluation.py --model bert-large-uncased-whole-word-masking-finetuned-squad
```

---

## Tips for Colab

1. **Session timeout:** Free Colab disconnects after ~12 hours of idle time. For training longer than that, use `tmux` or save checkpoints every epoch (already done).

2. **GPU quota:** Free users get ~12 hours of GPU per day. If you hit the limit, switch to CPU or wait 24 hours.

3. **Save frequently:** Always save to Google Drive — Colab deletes your `/content` folder when the session ends.

4. **Batch size:** On T4 GPU, you can use `batch_size=16` (vs. `batch_size=8` on CPU).